# 01 — Per-split Ensemble Fit + 시각화

**Step 1/4**: 데이터셋 로드 → 각 split 의 X_tr 만으로 K-member ensemble fit.

- 기존 ensemble 에 대한 의존성 없음 (데이터셋 직접 로드)
- 결과: `eval_dir/ensembles/split_{i}/c{c}/` 에 저장
- 시각화: per-member EBM heatmap, ensemble mean/std heatmap

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys, json, time, datetime as _dt
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

print('ready')

ready


## 1. 데이터셋 선택

In [ ]:
from ensemble_ebm import load_and_preprocess

# === 데이터셋 선택 (paper 14개 전체 지원) ===
# 형식: N samples, D features, C classes (raw → after <10 filter), cat=categorical cols
#
# 논문 primary 8 (OpenML):
# DATASET = 'stock'      # N=950,   D=9,   C=2             cat=0    주가 방향 (binary)
# DATASET = 'energy'     # N=768,   D=9,   C=37 → 23       cat=1    energy efficiency
# DATASET = 'collins'    # N=1000,  D=19,  C=30 → 26       cat=0    text classes
# DATASET = 'steel'      # N=1941,  D=33,  C=2             cat=0    steel-plate faults
DATASET = 'biodeg'       # N=1055,  D=41,  C=2             cat=0    QSAR biodegradation   <-- default
# DATASET = 'texture'    # N=5500,  D=40,  C=11            cat=0    image texture
# DATASET = 'fourier'    # N=2000,  D=76,  C=10            cat=0    digit Fourier descr
# DATASET = 'protein'    # N=1080,  D=77,  C=8             cat=0    protein localization
#
# 논문 extra 6 (paper "leakage-free UCI" — OpenML mirror):
# DATASET = 'statlog'    # N=1000,  D=20,  C=2             cat=13   German credit risk
# DATASET = 'support2'   # N=1470,  D=33,  C=2             cat=8    ICU mortality
# DATASET = 'auction'    # N=1470,  D=34,  C=2             cat=8    shill-bidding detection
# DATASET = 'abalone'    # N=4177,  D=8,   C=28 → 19       cat=1    age (rings)
# DATASET = 'mushroom'   # N=8124,  D=22,  C=2             cat=22   edible vs poisonous (all cat)
# DATASET = 'clinical'   # N=48790, D=14,  C=2             cat=8    clinical mortality (큰 dataset)

N_REAL = 100               # paper subset size ∈ {20, 50, 100, 200, 500}
# N_REAL = 20
# N_REAL = 50
# N_REAL = 200
# N_REAL = 500

# === Paper B.1 + B.3: class<10 filter + train-only preprocessing ===
# - class<10 filter (drop_small_classes=True): stratified 10-split 불가능한 rare class 제거
# - standardize=False, impute=False, encode_categorical=True: raw X (NaN 유지) + global OrdinalEncoder
# - per-split preprocessor 는 다음 셀에서 fit (paper B.3)
X_all, y_all, cat_idx = load_and_preprocess(
    DATASET, n_real=10**9, seed=42,
    standardize=False, impute=False,
    encode_categorical=True, return_cat_idx=True,
    drop_small_classes=True, min_class_count=10,    # paper B.1
)
CLASSES = sorted(np.unique(y_all).tolist())
_n_nan = int(np.isnan(X_all).sum())
print(f'dataset={DATASET}, full N={len(X_all)}, D={X_all.shape[1]}, NaN cells={_n_nan}')
print(f'classes {CLASSES} ({len(CLASSES)} kept after <10 filter), '
      f'bincount {[int((y_all==c).sum()) for c in CLASSES]}')
print(f'categorical columns: {cat_idx if cat_idx else "none (all numeric)"}')
print(f'X_all raw:  feature 0 mean={np.nanmean(X_all[:,0]):.2f}, '
      f'std={np.nanstd(X_all[:,0]):.2f}  (preprocessing 전)')
print(f'N_REAL (per-split train+val pool) = {N_REAL}')


## 3. Ensemble 하이퍼파라미터

각 파라미터를 별도 셀에서 조절합니다. 주석 처리된 대안들 참고.

In [ ]:
# === 3-1. 사용할 methods ===
ENSEMBLE_METHODS = ['Distance']
# ENSEMBLE_METHODS = ['Subsample']
# ENSEMBLE_METHODS = ['CornerNoise']
# ENSEMBLE_METHODS = ['NumFakeCorners']
# ENSEMBLE_METHODS = ['Distance', 'Subsample']
# ENSEMBLE_METHODS = ['Distance', 'CornerNoise']
# ENSEMBLE_METHODS = ['Distance', 'NumFakeCorners']
# ENSEMBLE_METHODS = ['Distance', 'Subsample', 'CornerNoise']
# ENSEMBLE_METHODS = ['Distance', 'Subsample', 'CornerNoise', 'NumFakeCorners']
print(f'ENSEMBLE_METHODS = {ENSEMBLE_METHODS}')

In [ ]:
# === 3-2. Distance 파라미터 ===
# mode: 'fixed'  → alpha 를 value 로 고정
#       'sweep'  → K 멤버에 dist_range 내 균등 배분
#       'random' → dist_range 내 랜덤
DISTANCE_PARAMS = {'mode': 'fixed', 'value': 5.0}
# DISTANCE_PARAMS = {'mode': 'fixed', 'value': 1.0}
# DISTANCE_PARAMS = {'mode': 'fixed', 'value': 3.0}
# DISTANCE_PARAMS = {'mode': 'fixed', 'value': 5.0}
# DISTANCE_PARAMS = {'mode': 'fixed', 'value': 10.0}
# DISTANCE_PARAMS = {'mode': 'fixed', 'value': 20.0}
# DISTANCE_PARAMS = {'mode': 'sweep', 'dist_range': [1.0, 5.0]}
# DISTANCE_PARAMS = {'mode': 'sweep', 'dist_range': [1.0, 10.0]}
# DISTANCE_PARAMS = {'mode': 'sweep', 'dist_range': [2.0, 20.0]}
# DISTANCE_PARAMS = {'mode': 'random', 'dist_range': [1.0, 10.0]}
print(f'DISTANCE_PARAMS = {DISTANCE_PARAMS}')

In [ ]:
# === 3-3. Subsample 파라미터 ===
# ratio: positive class 에서 subsample 비율
SUBSAMPLE_PARAMS = {'ratio': 0.25}
# SUBSAMPLE_PARAMS = {'ratio': 0.1}
# SUBSAMPLE_PARAMS = {'ratio': 0.3}
# SUBSAMPLE_PARAMS = {'ratio': 0.5}
# SUBSAMPLE_PARAMS = {'ratio': 0.75}
print(f'SUBSAMPLE_PARAMS = {SUBSAMPLE_PARAMS}')

In [ ]:
# === 3-4. CornerNoise 파라미터 ===
# noise_std_range: corner vertex 에 가우시안 노이즈 표준편차 범위
CORNER_NOISE_PARAMS = {'noise_std_range': [0.01, 0.3]}
# CORNER_NOISE_PARAMS = {'noise_std_range': [0.0, 0.1]}
# CORNER_NOISE_PARAMS = {'noise_std_range': [0.05, 0.5]}
# CORNER_NOISE_PARAMS = {'noise_std_range': [0.1, 1.0]}
print(f'CORNER_NOISE_PARAMS = {CORNER_NOISE_PARAMS}')

In [ ]:
# === 3-5. NumFakeCorners 파라미터 ===
# n_corners_range: 멤버별 사용할 fake corner vertex 수 범위
NUM_FAKE_CORNERS_PARAMS = {'n_corners_range': [2, 8]}
# NUM_FAKE_CORNERS_PARAMS = {'n_corners_range': [1, 4]}
# NUM_FAKE_CORNERS_PARAMS = {'n_corners_range': [4, 16]}
# NUM_FAKE_CORNERS_PARAMS = {'n_corners_range': [2, 2]}  # 고정
print(f'NUM_FAKE_CORNERS_PARAMS = {NUM_FAKE_CORNERS_PARAMS}')

In [ ]:
# === 3-6. K (ensemble 멤버 수) ===
K = 10
# K = 5
# K = 15
# K = 20
# K = 30
print(f'K = {K}')

In [ ]:
# === 3-7. Shared corners ===
# False: 멤버마다 다른 corner vertex (d>2 에서 자연 diversity)
# True:  K 멤버가 같은 corner vertex 공유 (Subsample/Distance 로 diversity)
SHARED_CORNERS = False
# SHARED_CORNERS = True
print(f'SHARED_CORNERS = {SHARED_CORNERS}')

In [ ]:
# === method_params dict 조립 (선택한 methods 에 맞게 자동) ===
ENSEMBLE_METHOD_PARAMS = {}
if 'Distance' in ENSEMBLE_METHODS:
    ENSEMBLE_METHOD_PARAMS['Distance'] = DISTANCE_PARAMS
if 'Subsample' in ENSEMBLE_METHODS:
    ENSEMBLE_METHOD_PARAMS['Subsample'] = SUBSAMPLE_PARAMS
if 'CornerNoise' in ENSEMBLE_METHODS:
    ENSEMBLE_METHOD_PARAMS['CornerNoise'] = CORNER_NOISE_PARAMS
if 'NumFakeCorners' in ENSEMBLE_METHODS:
    ENSEMBLE_METHOD_PARAMS['NumFakeCorners'] = NUM_FAKE_CORNERS_PARAMS

print(f'methods={ENSEMBLE_METHODS}, K={K}, shared_corners={SHARED_CORNERS}')
print(f'params={json.dumps(ENSEMBLE_METHOD_PARAMS, indent=2)}')

## 4. Split + 출력 설정

In [ ]:
from ensemble_ebm import fit_preprocessor, apply_preprocessor

SEED = 42
N_SPLITS = 10
WORKERS = 10

# === Paper Figure 9 (B.2 Data Splitting) + B.3 (train-only preprocessing) ===
# 1) Fixed test: N_test = min(N/2, 500)  — 모든 split 공통, train/val 과 disjoint
# 2) Oracle pool = 나머지 (N - N_test)
# 3) 매 split: oracle 에서 N_REAL stratified sampling → 80/20 train/val
# 4) 매 split 의 tr 만으로 imputer + scaler fit (test 통계 절대 참조 안 함)

n_test = min(len(X_all) // 2, 500)

oracle_idx, te_idx_fixed = next(StratifiedShuffleSplit(
    n_splits=1, test_size=n_test, random_state=SEED
).split(X_all, y_all))

assert N_REAL <= len(oracle_idx), \
    f"N_REAL={N_REAL} > oracle pool size={len(oracle_idx)} (full={len(X_all)}, test={n_test})"

X_oracle = X_all[oracle_idx]
y_oracle = y_all[oracle_idx]

n_val = max(len(CLASSES), int(round(N_REAL * 0.2)))
n_train = N_REAL - n_val

splits = []
preprocessors = []
for i in range(N_SPLITS):
    sampled_local, _ = next(StratifiedShuffleSplit(
        n_splits=1, train_size=N_REAL, random_state=SEED + i
    ).split(X_oracle, y_oracle))
    sampled_global = oracle_idx[sampled_local]

    tr_local, val_local = next(StratifiedShuffleSplit(
        n_splits=1, test_size=n_val, random_state=SEED + i + 10_000
    ).split(X_all[sampled_global], y_all[sampled_global]))
    tr_g = sampled_global[tr_local]
    val_g = sampled_global[val_local]
    splits.append((tr_g, val_g))

    # Paper B.3: fit preprocessor on tr_g ONLY
    preprocessors.append(fit_preprocessor(X_all[tr_g]))

te_set = set(te_idx_fixed.tolist())
for i, (tr, val) in enumerate(splits):
    assert te_set.isdisjoint(tr.tolist()), f"split {i}: train ∩ test ≠ ∅"
    assert te_set.isdisjoint(val.tolist()), f"split {i}: val ∩ test ≠ ∅"
    assert set(tr.tolist()).isdisjoint(val.tolist()), f"split {i}: train ∩ val ≠ ∅"

print(f'Full N={len(X_all)}, fixed test N={n_test}, oracle pool N={len(oracle_idx)}')
print(f'{N_SPLITS} splits — per-split: train={n_train}, val={n_val}, test={n_test} (fixed)')
print(f'Per-split train-only preprocessor fitted (imputer + z-score).')
_p0 = preprocessors[0]
print(f'  split 0 stats:  scl_mean[0]={_p0["scl_mean"][0]:+.3f}  '
      f'scl_std[0]={_p0["scl_std"][0]:.3f}  imp_mean[0]={_p0["imp_mean"][0]:+.3f}')
print(f'  → split 마다 scaler 가 다름 — 같은 te_fixed 가 split 별로 다른 스케일로 변환됨 (paper 방식).')


In [ ]:
def _method_tag():
    """활성화된 method 만 약식 태그로 조립."""
    parts = []
    if 'Distance' in ENSEMBLE_METHODS:
        p = DISTANCE_PARAMS
        if p['mode'] == 'fixed':
            parts.append(f"Dist-fix{p['value']:g}")
        elif p['mode'] == 'sweep':
            lo, hi = p['dist_range']
            parts.append(f"Dist-sw{lo:g}-{hi:g}")
        elif p['mode'] == 'random':
            lo, hi = p['dist_range']
            parts.append(f"Dist-rnd{lo:g}-{hi:g}")
    if 'Subsample' in ENSEMBLE_METHODS:
        parts.append(f"Sub-r{SUBSAMPLE_PARAMS['ratio']:g}")
    if 'CornerNoise' in ENSEMBLE_METHODS:
        lo, hi = CORNER_NOISE_PARAMS['noise_std_range']
        parts.append(f"Corn-{lo:g}-{hi:g}")
    if 'NumFakeCorners' in ENSEMBLE_METHODS:
        lo, hi = NUM_FAKE_CORNERS_PARAMS['n_corners_range']
        parts.append(f"NFC-{lo}-{hi}")
    return '_'.join(parts) if parts else 'nomethod'

method_tag = _method_tag()

ts = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
EVAL_DIR = Path(f'experiments/fair_eval/{ts}_{DATASET}_n{N_REAL}_K{K}_{method_tag}')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# data.npz: RAW X_all (NaN intact) — downstream 이 splits.npz preprocessor 로 변환
np.savez(EVAL_DIR / 'data.npz', X_all=X_all, y_all=y_all)

# splits.npz: indices + per-split preprocessor params (imputer + scaler)
split_kv = {'te_fixed': te_idx_fixed, 'oracle_idx': oracle_idx}
for i, (tr, val) in enumerate(splits):
    split_kv[f'tr_{i}'] = tr
    split_kv[f'val_{i}'] = val
    split_kv[f'imp_mean_{i}'] = preprocessors[i]['imp_mean']
    split_kv[f'scl_mean_{i}'] = preprocessors[i]['scl_mean']
    split_kv[f'scl_std_{i}']  = preprocessors[i]['scl_std']
np.savez(EVAL_DIR / 'splits.npz', **split_kv)

config = {
    'dataset': DATASET,
    'n_real': N_REAL,
    'methods': ENSEMBLE_METHODS,
    'method_params': ENSEMBLE_METHOD_PARAMS,
    'method_tag': method_tag,
    'shared_corners': SHARED_CORNERS, 'K': K,
    'seed': SEED, 'n_splits': N_SPLITS,
    'n_test': n_test, 'n_train': n_train, 'n_val': n_val,
    'classes': CLASSES,
    'n_samples': len(X_all), 'n_features': int(X_all.shape[1]),
    'split_protocol': 'paper Figure 9 — fixed test + oracle pool, 80/20 train/val',
    'class_filter': 'paper B.1 — classes with <10 samples removed',
    'preprocessing': 'paper B.3 — per-split train-only imputer + z-score',
    'encoding': 'paper run_experiment.py — global OrdinalEncoder(unknown_value=-1) for categorical',
    'categorical_columns': cat_idx,
    'data_storage': 'data.npz holds RAW X_all (NaN intact); splits.npz holds preprocessor params',
    'created_at': _dt.datetime.now().isoformat(timespec='seconds'),
}
(EVAL_DIR / 'config.json').write_text(json.dumps(config, indent=2))
print(f'EVAL_DIR = {EVAL_DIR}')


## 5. Ensemble Fit (병렬)

In [ ]:
from fair_eval_worker import fit_one_split_ensemble
from ensemble_ebm import apply_preprocessor

ens_dir = EVAL_DIR / 'ensembles'
ens_dir.mkdir(exist_ok=True)

# Paper B.3: 각 split 마다 tr-only preprocessor 적용 → 전체 X_all 변환 → 그 위에서 ensemble fit.
# (worker 에서 X_all[tr] 로 X_tr 추출하기 때문에 X_all 전체를 미리 변환해 전달.)
tasks = []
for i, (tr, _) in enumerate(splits):
    X_all_s = apply_preprocessor(X_all, preprocessors[i])
    tasks.append((i, tr, X_all_s, y_all, CLASSES,
                  ENSEMBLE_METHODS, ENSEMBLE_METHOD_PARAMS, SHARED_CORNERS,
                  K, SEED, str(ens_dir), DATASET, N_REAL))

t0 = time.time()
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(fit_one_split_ensemble, t): t[0] for t in tasks}
    for f in as_completed(futs):
        r = f.result()
        tag = '(cached)' if r.get('cached') else f'{r["dt"]:.1f}s'
        print(f'  split {r["split_i"]:>2d}  {tag}', flush=True)
print(f'\nDone -- {time.time()-t0:.1f}s')


## 6. 검증

In [ ]:
for i in range(N_SPLITS):
    sp = ens_dir / f'split_{i}'
    ok = all((sp / f'c{c}' / 'meta.json').exists() for c in CLASSES)
    n_ebms = sum(1 for p in (sp / 'c0').iterdir() if p.name.startswith('ebm_')) if ok else 0
    print(f'  split_{i}: {"OK" if ok and n_ebms==K else "FAIL"}  (K={n_ebms})')
print(f'\nEnsemble 저장 위치: {ens_dir}')
print(f'EVAL_DIR = {EVAL_DIR}  --> 2번 파일에 입력')

---
## 7. Heatmap 시각화 (standalone 가능)

**중요**: Section 1-6 재실행 없이 **기존 ensemble 폴더 로드** 로 바로 heatmap 그릴 수 있음.  
아래 7-0 셀에서 `EVAL_DIR` 만 바꿔주면 커널 재시작 후에도 OK.

- 7-0 (standalone 로더): 폴더 선택 + data/splits/ens_dir 복원
- 7-viz 설정: split 선택 + 3 viz on/off 스위치
- 7-1 Per-member heatmap
- 7-2 Mean + Std heatmap
- 7-3 축-정렬 2D slice (corner-pinned)

In [ ]:
# === 7-0a. Setup + 사용 가능한 폴더 목록 ===
# 커널 재시작 후 여기부터 돌리면 OK.

import os, sys, json, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA as _PCA

# --- 사용 가능한 EVAL_DIR 목록 ---
fair_root = Path('experiments/fair_eval')
print(f'{"폴더명":<55}  K    splits  ens  [method_tag]')
print('-' * 100)
if fair_root.exists():
    for p in sorted(fair_root.iterdir()):
        if p.is_dir() and (p / 'config.json').exists():
            cfg = json.loads((p / 'config.json').read_text())
            ens_ok = (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
            tag = cfg.get('method_tag', 'n/a')
            print(f'  {p.name:<53}  {cfg["K"]:<3d}  {cfg["n_splits"]:<6d} '
                  f'{"OK" if ens_ok else "--":<4}  [{tag}]')
else:
    print('  (experiments/fair_eval 없음 — 01 먼저 실행)')

In [ ]:
# === 7-0b. EVAL_DIR 선택 ===
# 위 목록에서 원하는 폴더 이름 복사해서 아래 한 줄 바꿔주세요.

EVAL_DIR = Path('experiments/fair_eval/20260419_182917_biodeg_n100_K10_Dist-fix5')   # <-- 변경

assert (EVAL_DIR / 'config.json').exists(), f'없음: {EVAL_DIR}'
assert (EVAL_DIR / 'ensembles' / 'split_0').exists(), '01 ensemble 없음'
print(f'선택: {EVAL_DIR.name}')

In [ ]:
# === 7-0c. 데이터 로드 + viz helper 정의 ===
config = json.loads((EVAL_DIR / 'config.json').read_text())
K        = config['K']
CLASSES  = config['classes']
N_SPLITS = config['n_splits']
DATASET  = config['dataset']
N_REAL   = config['n_real']

data = np.load(EVAL_DIR / 'data.npz')
X_all, y_all = data['X_all'], data['y_all']

sp = np.load(EVAL_DIR / 'splits.npz')
splits = [(sp[f'tr_{i}'], sp[f'val_{i}']) for i in range(N_SPLITS)]
te_idx_fixed = sp['te_fixed']

ens_dir     = EVAL_DIR / 'ensembles'
heatmap_dir = EVAL_DIR / 'heatmaps'
heatmap_dir.mkdir(exist_ok=True)

print(f'EVAL_DIR: {EVAL_DIR.name}')
print(f'  K={K}, classes={CLASSES}, n_splits={N_SPLITS}, dataset={DATASET}, n_real={N_REAL}')
print(f'  X_all {X_all.shape}, fixed test N={len(te_idx_fixed)}')

# === viz data helper — 7-1 / 7-2 가 각자 호출 (memory + disk 캐시) ===
_viz_cache = {}


def _compute_viz_one(si, c, VIZ_PAD=7.0, VIZ_H=0.1, VIZ_GPUS=(0, 1, 2, 3)):
    key = (int(si), int(c), float(VIZ_PAD), float(VIZ_H))
    if key in _viz_cache:
        return _viz_cache[key]

    from fair_eval_worker import compute_member_energy
    class_dir = ens_dir / f'split_{si}' / f'c{c}'
    cache_path = class_dir / f'heatmap_pad{VIZ_PAD}_h{VIZ_H}.npz'

    if cache_path.exists():
        cache = np.load(cache_path)
        K_actual = int(cache['K'])
        E_members = [cache[f'E_{k}'] for k in range(K_actual)]
        ZZ1, ZZ2 = cache['ZZ1'], cache['ZZ2']
    else:
        cd = np.load(class_dir / 'class_data.npz')
        X_class = cd['X_class']
        meta = json.loads((class_dir / 'meta.json').read_text())
        K_actual = meta['n_ebms']
        pca = _PCA(n_components=2).fit(X_class)
        Z_class = pca.transform(X_class)
        z_min = Z_class.min(0) - VIZ_PAD
        z_max = Z_class.max(0) + VIZ_PAD
        zz1 = np.arange(z_min[0], z_max[0], VIZ_H)
        zz2 = np.arange(z_min[1], z_max[1], VIZ_H)
        ZZ1, ZZ2 = np.meshgrid(zz1, zz2)
        grid_2d = np.column_stack([ZZ1.ravel(), ZZ2.ravel()])
        X_grid_d = pca.inverse_transform(grid_2d).astype(np.float64)

        tasks = [(str(class_dir), k, X_grid_d, list(VIZ_GPUS)[k % len(VIZ_GPUS)])
                 for k in range(K_actual)]
        print(f'  split {si} class {c}: computing K={K_actual} on {len(VIZ_GPUS)} GPUs...',
              end='', flush=True)
        t0 = time.time()
        energies = [None] * K_actual
        with ProcessPoolExecutor(max_workers=len(VIZ_GPUS)) as ex:
            futs = {ex.submit(compute_member_energy, t): t[1] for t in tasks}
            for f in as_completed(futs):
                k, energy = f.result()
                energies[k] = energy.reshape(ZZ1.shape)
        E_members = energies
        print(f' {time.time()-t0:.1f}s', flush=True)

        cache_kv = {'ZZ1': ZZ1, 'ZZ2': ZZ2, 'K': np.array(K_actual),
                     'Z_class': pca.transform(X_class),
                     'var_ratio': np.array(pca.explained_variance_ratio_.sum())}
        for k in range(K_actual):
            cache_kv[f'E_{k}'] = E_members[k]
        np.savez(cache_path, **cache_kv)

    neg_parts = []
    for k in range(K_actual):
        neg_path = class_dir / f'ebm_{k}' / 'negatives.npz'
        if neg_path.exists():
            neg_parts.append(np.load(neg_path)['X_neg'])
    X_class_raw = np.load(class_dir / 'class_data.npz')['X_class']
    pca_fit = _PCA(n_components=2).fit(X_class_raw)
    Z_neg_per_member = [pca_fit.transform(X_n) for X_n in neg_parts]
    X_neg_union = np.vstack(neg_parts) if neg_parts else np.empty((0, X_class_raw.shape[1]))
    Z_neg_union = pca_fit.transform(X_neg_union) if len(X_neg_union) else np.empty((0, 2))

    result = {
        'hm': {'ZZ1': ZZ1, 'ZZ2': ZZ2},
        'E_members': E_members,
        'Z_real': pca_fit.transform(X_class_raw),
        'Z_neg': Z_neg_union,
        'Z_neg_per_member': Z_neg_per_member,
        'pca': pca_fit,
    }
    _viz_cache[key] = result
    return result


def ensure_viz_data(splits_wanted, classes_wanted, VIZ_PAD=7.0, VIZ_H=0.1, VIZ_GPUS=(0, 1, 2, 3)):
    return {si: {c: _compute_viz_one(si, c, VIZ_PAD, VIZ_H, VIZ_GPUS) for c in classes_wanted}
            for si in splits_wanted}


print('✓ helper 정의 완료 — 7-1 / 7-2 / 7-3 각자 독립 실행 가능')

In [ ]:
# === 시각화 설정 (7-1, 7-2 공유) ===
VIZ_SPLITS = None              # None = 전체, 숫자/리스트 = 해당만
# VIZ_SPLITS = 0
# VIZ_SPLITS = [0, 3, 7]

VIZ_GPUS  = [0, 1, 2, 3]
VIZ_PAD   = 7.0
VIZ_H     = 0.1

# 그림 크기 (높이만; 가로는 종횡비 자동)
MEMBER_CELL_HEIGHT   = 4.5
SUMMARY_PANEL_HEIGHT = 6.5

# split 목록
if VIZ_SPLITS is None:
    _viz_splits = list(range(N_SPLITS))
elif isinstance(VIZ_SPLITS, int):
    _viz_splits = [VIZ_SPLITS]
else:
    _viz_splits = list(VIZ_SPLITS)

print(f'{len(_viz_splits)} splits to visualize: {_viz_splits}')

### 7-1. Per-member EBM heatmap

In [ ]:
# 독립 실행 가능 — 7-0 로더 + viz 설정 셀 실행했으면 바로 돌아감.
all_viz = ensure_viz_data(_viz_splits, CLASSES, VIZ_PAD, VIZ_H, VIZ_GPUS)

for si in _viz_splits:
    for c in CLASSES:
        v = all_viz[si][c]
        hm = v['hm']
        E_list = v['E_members']
        K_actual = len(E_list)
        Z_neg_per_member = v.get('Z_neg_per_member', [None] * K_actual)

        x_min, x_max = float(hm['ZZ1'].min()), float(hm['ZZ1'].max())
        y_min, y_max = float(hm['ZZ2'].min()), float(hm['ZZ2'].max())
        data_aspect = (x_max - x_min) / (y_max - y_min)
        cell_w = MEMBER_CELL_HEIGHT * data_aspect

        ncols = min(5, K_actual)
        nrows = (K_actual + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(cell_w * ncols, MEMBER_CELL_HEIGHT * nrows),
                                  constrained_layout=True)
        if K_actual == 1:
            axes = np.array([axes])
        axes_flat = axes.flat

        vmin = min(e.min() for e in E_list)
        vmax = max(e.max() for e in E_list)

        for k in range(K_actual):
            ax = axes_flat[k]
            ax.contourf(hm['ZZ1'], hm['ZZ2'], E_list[k], levels=20,
                        cmap='viridis', vmin=vmin, vmax=vmax)
            ax.scatter(*v['Z_real'].T, c='red', edgecolor='white', s=12,
                       linewidth=0.3, zorder=3, label='pos')
            Z_neg_k = Z_neg_per_member[k] if Z_neg_per_member[k] is not None else v['Z_neg']
            ax.scatter(*Z_neg_k.T, c='white', s=30, marker='x',
                       linewidths=1.2, zorder=3, label='neg')
            ax.set_title(f'{k}', fontsize=10)
            ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
            ax.set_aspect('equal')
            ax.tick_params(labelsize=6)
            if k == 0:
                ax.legend(fontsize=7, loc='upper left')

        for k in range(K_actual, nrows * ncols):
            axes_flat[k].axis('off')

        fig.suptitle(f'c{c}  (K={K_actual}, split {si})', fontsize=13)
        save_path = heatmap_dir / f'members_split{si}_c{c}.png'
        fig.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.show()

    print(f'  split {si} saved', flush=True)

print(f'\n{len(_viz_splits) * len(CLASSES)} per-member heatmaps → {heatmap_dir}')

### 7-2. Ensemble Mean + Std heatmap

In [ ]:
# 독립 실행 가능 — 7-0 로더 + viz 설정 셀 실행했으면 바로 돌아감.
all_viz = ensure_viz_data(_viz_splits, CLASSES, VIZ_PAD, VIZ_H, VIZ_GPUS)

for si in _viz_splits:
    for c in CLASSES:
        v = all_viz[si][c]
        hm = v['hm']
        E_list = v['E_members']
        E_stack = np.stack(E_list)
        E_mean = E_stack.mean(axis=0)
        E_std = E_stack.std(axis=0, ddof=1)

        x_min, x_max = float(hm['ZZ1'].min()), float(hm['ZZ1'].max())
        y_min, y_max = float(hm['ZZ2'].min()), float(hm['ZZ2'].max())
        data_aspect = (x_max - x_min) / (y_max - y_min)
        panel_w = SUMMARY_PANEL_HEIGHT * data_aspect
        fig, (ax1, ax2) = plt.subplots(
            1, 2,
            figsize=(panel_w * 2 * 1.18, SUMMARY_PANEL_HEIGHT),
            constrained_layout=True,
        )

        for ax, data, cmap, title in [
            (ax1, E_mean, 'viridis', f'c{c} — Mean Energy (split {si})'),
            (ax2, E_std,  'magma',   f'c{c} — Std (diversity) (split {si})'),
        ]:
            im = ax.contourf(hm['ZZ1'], hm['ZZ2'], data, levels=20, cmap=cmap)
            ax.scatter(*v['Z_real'].T, c='red', edgecolor='white', s=14,
                       linewidth=0.3, zorder=3, label='pos')
            ax.scatter(*v['Z_neg'].T, c='white', s=35, marker='x',
                       linewidths=1.2, zorder=3, label='neg')
            ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
            ax.set_aspect('equal')
            ax.set_title(title, fontsize=12)
            ax.legend(fontsize=9)
            fig.colorbar(im, ax=ax, shrink=0.8)

        fig.savefig(heatmap_dir / f'mean_std_split{si}_c{c}.png',
                    dpi=120, bbox_inches='tight')
        plt.show()

    print(f'  split {si} saved', flush=True)

print(f'\nmean+std heatmaps → {heatmap_dir}')

### 7-3. 축-정렬 2D slice (PCA 말고 실제 feature 축)

원하는 `split`, `class`, `k`(여러 개 가능) 에 대해, feature 2 개를 선택해 그 축 위에서 energy 를 계산.  
- **PCA slice 와 차이**: PCA 는 2D 평면으로 선형 투영 후 역변환 — negative 위치가 왜곡됨.  
- **축-정렬 slice**: 선택한 2 feature 만 변화, 나머지 39 dim = positives mean (≈0 z-score). negative 의 두 좌표는 **그대로 ±distance**.

In [ ]:
# 독립 실행 가능 — 7-0 로더 실행했으면 바로 돌아감. all_viz 필요 없음 (자체 compute).
# === 선택 ===
SLICE_SPLIT  = 0
SLICE_CLASS  = 0
SLICE_K_LIST = [0, 1, 2]         # 시각화할 멤버 k

SLICE_DIMS   = 'auto_corners'    # 각 k 의 4 neg 이 4 꼭짓점 되는 (i,j) 자동
# SLICE_DIMS = 'auto_var'        # positives variance 상위 2 dim (전 k 공유)
# SLICE_DIMS = (0, 1)            # 수동 지정

SLICE_PAD    = 2.0
SLICE_H      = 0.2
SLICE_GPU    = 0

PRINT_POS_DATA = True            # True: 선택 dim 에서 positives 값 출력
PRINT_POS_HEAD = 20              # 출력할 positives 개수

from fair_eval_worker import compute_member_energy

slice_class_dir = ens_dir / f'split_{SLICE_SPLIT}' / f'c{SLICE_CLASS}'
X_class_raw = np.load(slice_class_dir / 'class_data.npz')['X_class']
d = X_class_raw.shape[1]

# === feature 이름 로드 (OpenML 에서, 실패하면 idx 로 fallback) ===
def _load_feature_names(dataset_name):
    try:
        import openml
        from ensemble_ebm import DATASET_IDS
        oid = DATASET_IDS.get(dataset_name)
        if oid is None:
            return None
        ds = openml.datasets.get_dataset(oid,
                                          download_data=False,
                                          download_qualities=False,
                                          download_features_meta_data=True)
        _, _, _, names = ds.get_data(target=ds.default_target_attribute)
        return list(names)
    except Exception:
        return None

feature_names = _load_feature_names(DATASET)
def _fname(i):
    if feature_names and i < len(feature_names):
        return f'{i}:{feature_names[i]}'
    return f'{i}'

if feature_names:
    print(f'dataset={DATASET}  →  {len(feature_names)} feature names loaded')
else:
    print(f'dataset={DATASET}  →  feature names 없음 (idx 로 표시)')

negs_picked = {}
for k in SLICE_K_LIST:
    negs_picked[k] = np.load(slice_class_dir / f'ebm_{k}' / 'negatives.npz')['X_neg']


def find_antipodal_pairs(X_neg):
    for i in range(1, 4):
        if np.allclose(X_neg[0], -X_neg[i]):
            others = [j for j in range(4) if j not in (0, i)]
            return X_neg[0], X_neg[others[0]]
    return None, None


def find_corner_dims(v, w):
    eq = (np.sign(v) == np.sign(w))
    true_idx = np.where(eq)[0]
    false_idx = np.where(~eq)[0]
    if len(true_idx) and len(false_idx):
        return int(true_idx[0]), int(false_idx[0])
    return None, None


shared_dims = None
dims_per_k = {}
if SLICE_DIMS == 'auto_corners':
    for k in SLICE_K_LIST:
        v, w = find_antipodal_pairs(negs_picked[k])
        if v is None:
            raise RuntimeError(f'k={k} negatives 가 antipodal 구조 아님')
        i, j = find_corner_dims(v, w)
        if i is None:
            raise RuntimeError(f'k={k}: 4-corner 를 만드는 (i,j) 못 찾음')
        dims_per_k[k] = (i, j)
        print(f'  k={k}: dims=({_fname(i)}, {_fname(j)})  neg 4점: '
              f'({v[i]:+.0f},{v[j]:+.0f}) ({-v[i]:+.0f},{-v[j]:+.0f}) '
              f'({w[i]:+.0f},{w[j]:+.0f}) ({-w[i]:+.0f},{-w[j]:+.0f})')
elif SLICE_DIMS == 'auto_var':
    var = X_class_raw.var(0)
    shared_dims = tuple(sorted(np.argsort(var)[-2:].tolist()))
    print(f'공유 dims: ({_fname(shared_dims[0])}, {_fname(shared_dims[1])})')
else:
    shared_dims = tuple(SLICE_DIMS)
    print(f'공유 dims: ({_fname(shared_dims[0])}, {_fname(shared_dims[1])})')
if shared_dims is not None:
    for k in SLICE_K_LIST:
        dims_per_k[k] = shared_dims

# === positives 좌표 출력 ===
if PRINT_POS_DATA:
    print(f'\n{"="*78}')
    print(f'Positives 좌표 (원본 feature 값 — 변환 없음)')
    print(f'  X_class shape: {X_class_raw.shape}  (n_positives = {len(X_class_raw)})')
    print('='*78)
    unique_dims = set(dims_per_k.values())
    for dims in sorted(unique_dims):
        di, dj = dims
        ks_sharing = [k for k in SLICE_K_LIST if dims_per_k[k] == dims]
        print(f'\ndims = ({_fname(di)}, {_fname(dj)})  [k={ks_sharing}]')
        print(f'  feature {_fname(di)}:  mean={X_class_raw[:, di].mean():+.3f}  '
              f'std={X_class_raw[:, di].std():.3f}  '
              f'min={X_class_raw[:, di].min():+.3f}  max={X_class_raw[:, di].max():+.3f}')
        print(f'  feature {_fname(dj)}:  mean={X_class_raw[:, dj].mean():+.3f}  '
              f'std={X_class_raw[:, dj].std():.3f}  '
              f'min={X_class_raw[:, dj].min():+.3f}  max={X_class_raw[:, dj].max():+.3f}')
        head = min(PRINT_POS_HEAD, len(X_class_raw))
        print(f'  처음 {head}개 positives (x=feature {di}, y=feature {dj}):')
        for i in range(head):
            print(f'    pos[{i:>3d}]:  ({X_class_raw[i, di]:+.3f}, {X_class_raw[i, dj]:+.3f})')
        print(f'  neg 4 corners (각 k 별):')
        for k in ks_sharing:
            for ni, n in enumerate(negs_picked[k]):
                print(f'    k={k}  neg[{ni}]:  ({n[di]:+.1f}, {n[dj]:+.1f})')
    print(f'{"="*78}\n')

t0 = time.time()
X_ref = X_class_raw.mean(0)
slice_data = {}
for k in SLICE_K_LIST:
    di, dj = dims_per_k[k]
    all_x = np.concatenate([X_class_raw[:, di], negs_picked[k][:, di]])
    all_y = np.concatenate([X_class_raw[:, dj], negs_picked[k][:, dj]])
    x_min, x_max = all_x.min() - SLICE_PAD, all_x.max() + SLICE_PAD
    y_min, y_max = all_y.min() - SLICE_PAD, all_y.max() + SLICE_PAD
    xs = np.arange(x_min, x_max, SLICE_H)
    ys = np.arange(y_min, y_max, SLICE_H)
    XX, YY = np.meshgrid(xs, ys)

    X_grid_d = np.tile(X_ref, (XX.size, 1))
    X_grid_d[:, di] = XX.ravel()
    X_grid_d[:, dj] = YY.ravel()

    _, energy = compute_member_energy(
        (str(slice_class_dir), k, X_grid_d.astype(np.float64), SLICE_GPU))
    slice_data[k] = {
        'XX': XX, 'YY': YY, 'E': energy.reshape(XX.shape),
        'dims': (di, dj), 'x_min': x_min, 'x_max': x_max, 'y_min': y_min, 'y_max': y_max,
    }
print(f'energy 계산 {time.time()-t0:.1f}s')

n = len(SLICE_K_LIST)
ncols = min(5, n)
nrows = (n + ncols - 1) // ncols

_H = MEMBER_CELL_HEIGHT if 'MEMBER_CELL_HEIGHT' in dir() else 4.5

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(_H * ncols, _H * nrows),
                          constrained_layout=True)
if n == 1:
    axes = np.array([axes])
axes_flat = np.array(axes).flat

vmin = min(s['E'].min() for s in slice_data.values())
vmax = max(s['E'].max() for s in slice_data.values())

for i, k in enumerate(SLICE_K_LIST):
    s = slice_data[k]
    di, dj = s['dims']
    ax = axes_flat[i]
    ax.contourf(s['XX'], s['YY'], s['E'], levels=20, cmap='viridis', vmin=vmin, vmax=vmax)
    ax.scatter(X_class_raw[:, di], X_class_raw[:, dj],
               c='red', edgecolor='white', s=14, linewidth=0.3, zorder=3, label='pos')
    ax.scatter(negs_picked[k][:, di], negs_picked[k][:, dj],
               c='white', s=80, marker='x', linewidths=1.8, zorder=3, label='neg (4 corners)')
    ax.set_title(f'k={k}  ({_fname(di)}, {_fname(dj)})', fontsize=10)
    ax.set_xlim(s['x_min'], s['x_max']); ax.set_ylim(s['y_min'], s['y_max'])
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7)
    ax.set_xlabel(_fname(di), fontsize=9)
    ax.set_ylabel(_fname(dj), fontsize=9)
    if i == 0:
        ax.legend(fontsize=8, loc='upper right')

for i in range(n, nrows * ncols):
    axes_flat[i].axis('off')

fig.suptitle(
    f'Corner-pinned 2D slice — split {SLICE_SPLIT}, class {SLICE_CLASS}',
    fontsize=12)
save_path = heatmap_dir / f'corner_slice_split{SLICE_SPLIT}_c{SLICE_CLASS}.png'
fig.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'saved: {save_path}')